In [15]:
"""
exp03 Step 01: "Sell At Strength" Exploration (all-time-high exits; EXPLORATION ON 2005-2014 ONLY)

Hypothesis (Nicolas, 2026-10-04): instead of selling on weakness (exp02 stops), sell when SPY is at or near its all-time
high and buy back later, lower. The same identity as exp02 applies: an episode that sells at S and buys back at R ends
with S / R times the shares, so the idea wins only if prices AFTER near-ATH days tend to FALL (not merely rise less).

This notebook repeats the exploration of 2026-10-04 (ath_exit_exploration / ath_exit_exploration_clean, 30 trials in
the legacy trial log) under this experiment's name, on 2005-01-03 -> 2014-12-31 only (training data of every fold; the
data is cut at so.config.EXPLORATION_DATA_CUTOFF_DATE_STR = 2015-03-18, so no forward return reaches the walk-forward period):
    1. Forward returns after near-ATH days vs all days (n = 1 .. 60 sessions), monthly block bootstrap of the difference.
    2. The 15 rules of PROTOCOL.md §4 over the whole exploration period (one continuous window, start invested).

The walk-forward (step 02) uses ALL 15 rules as candidates: none is picked because of its result here.
Caveats: the "all-time high" is the highest high since 2005-01-03 (2005-2006 highs are pseudo-ATHs), and 2005-2014
contains only about 15 separate ATH episodes.
Prerequisite: the raw data only. Outputs: store04_experiments/exp03_ath_exit/step01_exploration_data/.
"""
None

In [16]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp03_ath_exit import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp03_ath_exit.rules import get_rule_candidate_list, simulate_rule_dict, get_rule_metric_dict, get_event_forward_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [17]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp03_ath_exit	Protocol:	1.0	Config Hash:	b8523207d1
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp03_ath_exit/step01_exploration_data/
ℹ️ No earlier 'exploration' entry of exp03_ath_exit in the trial log (config hash b8523207d1).


In [18]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tExploration window:\t{exploration_start_str} -> {exploration_end_str}")

Data:	2005-01-03 -> 2015-03-18	(2,568 sessions)	Exploration window:	2005-01-03 -> 2014-12-31


In [19]:
"""
Check 1: forward log return from the 15:59 fill of a near-ATH day to the 15:59 fill n sessions later, compared with the
same quantity for every day. diff < 0 = prices rise LESS than usual after near-ATH days; selling at S and buying back at R
pays only if the event mean itself is NEGATIVE.
"""
# DEFINE THE EVENTS (THE THREE EXIT THRESHOLDS AND THE 52-WEEK HIGH)
ath_dist_arr = daily_pdf["ath_drawdown_pct"].to_numpy(dtype=float)
high250_dist_arr = np.nan_to_num(daily_pdf["high250_dist_pct"].to_numpy(dtype=float), nan=-1.0)
event_dict = {**{f"ATH within {ath_within:.1%}": ath_dist_arr >= -ath_within for ath_within in exp_config.ATH_WITHIN_LIST},
              f"52-week high within {exp_config.HIGH250_WITHIN:.0%}": high250_dist_arr >= -exp_config.HIGH250_WITHIN}
# CALL FUNCTION TO BUILD THE FORWARD RETURN TABLE
forward_pdf = get_event_forward_pdf(daily_pdf, event_dict, exploration_end_str)
# DISPLAY THE NEAR-ATH DAYS PER YEAR
in_window_arr = np.array([date_object <= pd.Timestamp(exploration_end_str).date() for date_object in daily_pdf["date"]])
print("Days within 1% of the ATH per year:", pd.Series([d.year for d in daily_pdf["date"].to_numpy()[in_window_arr & (ath_dist_arr >= -0.01)]]).value_counts().sort_index().to_dict())
# PREVIEW DATAFRAME
forward_pdf.round(4)

Days within 1% of the ATH per year: {2005: 30, 2006: 81, 2007: 61, 2013: 101, 2014: 136}


,event,horizon,episode_count,event_day_count,event_mean_log_return,all_mean_log_return,diff,diff_ci_low,diff_ci_high,event_up_share,all_up_share
0,ATH within 0.5%,1,15,230,-0.0002,0.0002,-0.0004,-0.0012,0.0002,0.4913,0.5453
1,ATH within 0.5%,5,15,230,-0.0013,0.0011,-0.0024,-0.0058,0.0006,0.5000,0.5719
2,ATH within 0.5%,10,15,230,-0.0014,0.0021,-0.0035,-0.0094,0.0017,0.5870,0.6077
3,ATH within 0.5%,20,15,230,-0.0002,0.0043,-0.0045,-0.0126,0.0037,0.5522,0.6395
4,ATH within 0.5%,40,15,230,0.0045,0.0087,-0.0042,-0.0167,0.0080,0.6391,0.6586
5,ATH within 0.5%,60,15,230,0.0130,0.0131,-0.0000,-0.0154,0.0147,0.7696,0.6878
6,ATH within 1.0%,1,12,409,-0.0003,0.0002,-0.0005,-0.0012,0.0001,0.5110,0.5453
7,ATH within 1.0%,5,12,409,-0.0009,0.0011,-0.0020,-0.0047,0.0008,0.5257,0.5719
8,ATH within 1.0%,10,12,409,-0.0007,0.0021,-0.0028,-0.0076,0.0020,0.5868,0.6077
9,ATH within 1.0%,20,12,409,-0.0009,0.0043,-0.0052,-0.0132,0.0028,0.5599,0.6395


In [20]:
"""
Check 2: the 15 rules (one continuous window, start invested, same costs as exp02, forced buy-back after 60 decisions).
One of them doing well while its neighbours do badly is what noise looks like.
"""
# SIMULATE BUY AND HOLD
buy_hold_metric_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)["metric_dict"]
# LISTS TO HOLD THE RESULTS
rule_dict_list, simulation_dict_dict = [], {}
# ITERATE OVER THE RULES
for candidate_dict in get_rule_candidate_list():
    # SIMULATE THE RULE
    simulation_dict = simulate_rule_dict(ohlcv_array_dict, daily_pdf, exploration_start_str, exploration_end_str, candidate_dict)
    simulation_dict_dict[(candidate_dict["ath_within"], candidate_dict["rule"])] = simulation_dict
    # STORE THE METRICS
    rule_dict_list.append({"ath_within": candidate_dict["ath_within"], "rule": candidate_dict["rule"], **get_rule_metric_dict(simulation_dict, buy_hold_metric_dict["total_return"])})
# CONVERT TO A DATAFRAME
rule_pdf = pd.DataFrame(rule_dict_list)
# DISPLAY INFORMATION
print(f"Buy & hold:\ttotal return {buy_hold_metric_dict['total_return']:.2%} | Sharpe {buy_hold_metric_dict['sharpe_ratio']:.2f} | max drawdown {buy_hold_metric_dict['max_drawdown']:.2%}")
# DISPLAY THE EXCESS RETURN GRID (THRESHOLD x BUY-BACK)
display(rule_pdf.pivot(index="ath_within", columns="rule", values="excess_return").round(4))
# PREVIEW DATAFRAME
rule_pdf.round(4)

Buy & hold:	total return 69.02% | Sharpe 0.36 | max drawdown -56.44%


rule,delay_20,delay_5,delay_60,dip_2pct,dip_5pct
ath_within,,,,,
0.005,0.0495,0.0713,-0.0339,0.0265,0.1168
0.010,-0.0772,-0.0794,-0.1943,-0.1304,-0.0206
0.020,-0.2358,0.0101,-0.4693,-0.2410,-0.2989


,ath_within,rule,total_return,excess_return,sharpe_ratio,max_drawdown,exit_count,forced_reentry_count,cash_session_pct,mean_cash_sessions,bought_back_lower_pct,mean_log_share_gain
0,0.005,delay_5,0.7614,0.0713,0.3863,-0.5632,69,0,0.1372,5.0000,0.4638,0.0007
1,0.005,delay_20,0.7396,0.0495,0.3850,-0.5520,31,0,0.2465,20.0000,0.5161,0.0010
2,0.005,delay_60,0.6563,-0.0339,0.3656,-0.5231,16,0,0.3706,60.0000,0.3750,-0.0012
3,0.005,dip_2pct,0.7167,0.0265,0.3781,-0.5598,19,7,0.2911,38.3889,0.5789,0.0009
4,0.005,dip_5pct,0.8069,0.1168,0.4074,-0.5519,17,11,0.3280,49.0000,0.4118,0.0040
5,0.010,delay_5,0.6107,-0.0794,0.3415,-0.5597,91,0,0.1809,5.0000,0.4396,-0.0005
6,0.010,delay_20,0.6130,-0.0772,0.3463,-0.5600,37,0,0.2942,20.0000,0.3784,-0.0012
7,0.010,delay_60,0.4958,-0.1943,0.3097,-0.5299,16,0,0.3742,60.0000,0.3125,-0.0076
8,0.010,dip_2pct,0.5598,-0.1304,0.3277,-0.5601,18,8,0.3046,42.6471,0.5000,-0.0044
9,0.010,dip_5pct,0.6695,-0.0206,0.3651,-0.5517,17,11,0.3324,49.6875,0.3529,-0.0007


In [21]:
# PREVIEW THE EPISODES OF ONE RULE (WITHIN 1% OF THE ATH, BUY BACK 5% LOWER OR AFTER 60 DECISIONS)
simulation_dict_dict[(0.01, "dip_5pct")]["episode_pdf"][["exit_ts", "exit_fill_price", "reentry_ts", "reentry_fill_price", "cash_session_count", "reentry_reason", "share_gain_pct"]]

,exit_ts,exit_fill_price,reentry_ts,reentry_fill_price,cash_session_count,reentry_reason,share_gain_pct
0,2005-02-11 15:59:00-05:00,120.69,2005-04-15 15:59:00-04:00,114.31,43,dip,0.055813
1,2005-07-12 15:59:00-04:00,122.22,2005-10-05 15:59:00-04:00,119.64,60,forced,0.021565
2,2005-11-17 15:59:00-05:00,124.51,2006-02-15 15:59:00-05:00,128.17,60,forced,-0.028556
3,2006-02-16 15:59:00-05:00,129.03,2006-05-15 15:59:00-04:00,129.65,60,forced,-0.004782
4,2006-09-26 15:59:00-04:00,133.55,2006-12-20 15:59:00-05:00,142.11,60,forced,-0.060235
5,2006-12-27 15:59:00-05:00,142.55,2007-03-27 15:59:00-04:00,142.74,60,forced,-0.001331
6,2007-04-13 15:59:00-04:00,145.23,2007-07-11 15:59:00-04:00,151.78,60,forced,-0.043155
7,2007-07-12 15:59:00-04:00,154.61,2007-07-27 15:59:00-04:00,146.06,11,dip,0.058538
8,2007-10-01 15:59:00-04:00,154.46,2007-11-09 15:59:00-05:00,145.53,29,dip,0.061362
9,2013-03-11 15:59:00-04:00,156.01,2013-06-05 15:59:00-04:00,161.31,60,forced,-0.032856


In [22]:
# SAVE THE RESULTS
write_csv_file_to_path(forward_pdf, f"{output_path_str}forward_return_data.csv", "W")
write_csv_file_to_path(rule_pdf, f"{output_path_str}rule_data.csv", "W")
# LOG ONE EXPLORATION ENTRY PER RULE (NO VALIDATION OR TEST WINDOW IS TOUCHED)
logged_pdf = log_trial_pdf(exp_config, "exploration", rule_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["ath_within", "rule", "window"],
                           ["total_return", "excess_return", "sharpe_ratio", "max_drawdown", "exit_count", "mean_log_share_gain"], False,
                           note_str_in="exp03 step01 sell-at-strength exploration (2005-2014 only)")
print(f"Logged {len(logged_pdf)} exploration entries (config hash {get_config_hash_str(exp_config)})")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp03_ath_exit/step01_exploration_data/forward_return_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp03_ath_exit/step01_exploration_data/forward_return_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp03_ath_exit/step01_exploration_data/rule_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp03_ath_exit/step01_exploration_data/rule_data.csv' Has Been Saved!
Logged 15 exploration entries (config hash b8523207d1)


In [23]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
1,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
2,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
3,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
4,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
